# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedaMalaika75/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
**Method selected:** Logistic Regression used as a ranking/scoring model.

My project is a ranking task. The model will estimate the probability that a content item will show observed decline in search visibility. These probabilities will then be used to rank pages from highest to lowest opportunity.

I selected Logistic Regression because it is simple, interpretable, and suitable as the first learned model. Its ranking will be compared with my Week 4 baseline using the same data, the same split, and Precision@50.

In [4]:
from sklearn.linear_model import LogisticRegression

method_check = LogisticRegression(
    max_iter=2000,
    random_state=42
)

print("Selected method:", method_check.__class__.__name__)
print("Ranking score: predicted probability")
print("Evaluation metrics: Precision@20 and Precision@50")


Selected method: LogisticRegression
Ranking score: predicted probability
Evaluation metrics: Precision@20 and Precision@50


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
**Split selected:** 80% training and 20% testing, grouped by `client_id`.

All pages belonging to the same client will remain entirely in either the training set or the test set. This prevents the model from learning client-specific patterns during training and then seeing the same client again during testing.

I will use a fixed random seed of 42 so that the split can be reproduced. The Week 4 baseline and the Logistic Regression model will both be evaluated on this same test set using Precision@50.

In [5]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "SyedaMalaika75/flyrank-ml-internship/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

# Outcome label: used for evaluation, not as an input feature
df["decline_label"] = (
    df["trend_direction"] == "down"
).astype(int)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_index, test_index = next(
    splitter.split(
        df,
        y=df["decline_label"],
        groups=df["client_id"]
    )
)

train_df = df.iloc[train_index].copy()
test_df = df.iloc[test_index].copy()

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Training clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())
print("Clients appearing in both sets:", len(train_clients & test_clients))

assert len(train_clients & test_clients) == 0
print("Grouped split check passed.")


Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Clients appearing in both sets: 0
Grouped split check passed.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*
**Training and comparison plan:** I train a Logistic Regression model using safe content metadata and trailing 90-day performance signals.

The model produces a probability score for each content item. I rank the test items from highest to lowest probability of observed decline.

I exclude content and client IDs, provider and model names, the 30-day trend input columns, `trend_direction`, and `trend_pct`. The Week 4 rule baseline is recalculated on the same held-out clients and compared with the model using the same Precision@20 and Precision@50 metrics.

In [6]:
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# --------------------------------------------------
# PREPARE TRAIN AND TEST COPIES
# --------------------------------------------------

train_model_df = train_df.copy()
test_model_df = test_df.copy()

# avg_position = 0 means unavailable data, not rank zero.
for frame in [train_model_df, test_model_df]:
    frame["avg_position_missing"] = (
        frame["avg_position"] == 0
    ).astype(int)

    frame.loc[
        frame["avg_position"] == 0,
        "avg_position"
    ] = np.nan


# --------------------------------------------------
# SAFE FEATURES
# --------------------------------------------------

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "ctr",
    "avg_position",
    "avg_position_missing",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

feature_columns = numeric_features + categorical_features
target_column = "decline_label"


# --------------------------------------------------
# LEAKAGE CHECK
# --------------------------------------------------

banned_features = {
    "content_id",
    "client_id",
    "provider_used",
    "model_used",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
}

leaked_features = sorted(
    set(feature_columns) & banned_features
)

print("Leaked features detected:", leaked_features)

assert not leaked_features, (
    "Leakage detected in the selected features."
)


# --------------------------------------------------
# BUILD TRAINING MATRICES
# --------------------------------------------------

X_train = train_model_df[feature_columns]
y_train = train_model_df[target_column]

X_test = test_model_df[feature_columns]
y_test = test_model_df[target_column]


# --------------------------------------------------
# PREPROCESSING
# --------------------------------------------------

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        ),
    ]
)


# --------------------------------------------------
# TRAIN LOGISTIC REGRESSION
# --------------------------------------------------

model_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        ),
    ]
)

model_pipeline.fit(X_train, y_train)

model_scores = model_pipeline.predict_proba(
    X_test
)[:, 1]


# --------------------------------------------------
# REBUILD WEEK 4 BASELINE ON THE SAME TEST SET
# --------------------------------------------------

baseline_ranked = test_model_df.copy()

baseline_ranked["signal_stale_visible"] = (
    (
        baseline_ranked["days_since_last_update"]
        >= 180
    )
    & (
        baseline_ranked["impressions_90d"]
        >= 500
    )
)

baseline_ranked["signal_low_ctr_visible"] = (
    (
        baseline_ranked["impressions_90d"]
        >= 500
    )
    & (
        baseline_ranked["avg_position"]
        .between(1, 20)
    )
    & (
        baseline_ranked["ctr"] < 0.5
    )
)

baseline_ranked["volume_tiebreak"] = (
    np.minimum(
        baseline_ranked["impressions_90d"]
        / 5000,
        1.0
    )
    * 10
)

baseline_ranked["baseline_score"] = (
    baseline_ranked[
        "signal_stale_visible"
    ].astype(int) * 100
    + baseline_ranked[
        "signal_low_ctr_visible"
    ].astype(int) * 60
    + baseline_ranked[
        "volume_tiebreak"
    ]
)

baseline_ranked = baseline_ranked.sort_values(
    [
        "baseline_score",
        "impressions_90d"
    ],
    ascending=[False, False]
).reset_index(drop=True)


# --------------------------------------------------
# RANK MODEL RESULTS
# --------------------------------------------------

model_ranked = test_model_df[
    [
        "content_id",
        "decline_label",
        "impressions_90d"
    ]
].copy()

model_ranked["model_score"] = model_scores

model_ranked = model_ranked.sort_values(
    [
        "model_score",
        "impressions_90d"
    ],
    ascending=[False, False]
).reset_index(drop=True)


# --------------------------------------------------
# PRECISION AT K
# --------------------------------------------------

def precision_at_k(frame, k):
    return frame.head(k)[
        "decline_label"
    ].mean()


test_base_rate = y_test.mean()

baseline_p20 = precision_at_k(
    baseline_ranked,
    20
)

baseline_p50 = precision_at_k(
    baseline_ranked,
    50
)

model_p20 = precision_at_k(
    model_ranked,
    20
)

model_p50 = precision_at_k(
    model_ranked,
    50
)


# --------------------------------------------------
# REQUIRED COMPARISON TABLE
# --------------------------------------------------

comparison_table = pd.DataFrame(
    {
        "Approach": [
            "Week 4 rule baseline",
            "Logistic Regression"
        ],
        "Test base rate": [
            test_base_rate,
            test_base_rate
        ],
        "Precision@20": [
            baseline_p20,
            model_p20
        ],
        "Precision@50": [
            baseline_p50,
            model_p50
        ],
    }
)

print("Training rows:", len(train_model_df))
print("Testing rows:", len(test_model_df))
print("Features used:", len(feature_columns))

display(
    comparison_table.round(3)
)

print("\nTop 10 model-ranked items:")

display(
    model_ranked[
        [
            "content_id",
            "model_score",
            "decline_label",
            "impressions_90d"
        ]
    ].head(10)
)


Leaked features detected: []
Training rows: 23837
Testing rows: 6163
Features used: 32


,Approach,Test base rate,Precision@20,Precision@50
0,Week 4 rule baseline,0.511,0.4,0.42
1,Logistic Regression,0.511,0.7,0.72



Top 10 model-ranked items:


,content_id,model_score,decline_label,impressions_90d
0,content_a928cb66d230,0.928992,1,128
1,content_96dba8ca02c1,0.922465,1,138
2,content_b5e9e6453511,0.915574,1,157
3,content_8ede62882d0b,0.906450,1,556
4,content_374e795aab68,0.900690,0,235
5,content_453722754fea,0.896446,1,140079
6,content_41baf0722ad9,0.893844,0,3115
7,content_7be5f150dc65,0.893454,0,290
8,content_5d77d3077984,0.887680,1,68
9,content_ff102de380d8,0.885448,1,149


### Error interpretation

The model correctly identified 36 actual decline cases among its top 50 recommendations, giving a Precision@50 of 0.72.

However, approximately 14 items in the top 50 are false positives. The examples show that a high model score does not guarantee that the page will actually decline. Some pages can have historical patterns similar to declining content while still maintaining or improving their visibility.

There are also actual decline cases ranked below the top 50. These false-negative-style ranking errors show that the current feature set does not capture every cause of decline.

Overall, the learned model provides a substantial improvement over the rule baseline, but human review is still useful before turning ranked recommendations into actions.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
**Error analysis and interpretation**

The Logistic Regression model clearly outperformed the Week 4 rule-based baseline on the same held-out test set. Precision@20 increased from 0.40 to 0.70, while Precision@50 increased from 0.42 to 0.72.

This means that 72% of the first 50 pages ranked by the learned model were actual decline cases, compared with only 42% for the rule baseline.

The model is not perfect. Some highly ranked pages are false positives: the model assigns them a high probability of decline even though their observed decline label is 0. These cases are useful because they show where the learned patterns do not match the final outcome.

I will inspect false positives among the highest-ranked pages and false negatives that the model ranked too low. This helps identify the types of cases where the model is making mistakes rather than judging performance from a single metric alone.

In [7]:
# ---------------------------------------------
# ERROR ANALYSIS
# ---------------------------------------------

analysis_df = test_model_df[
    [
        "content_id",
        "decline_label",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "days_since_last_update"
    ]
].copy()

analysis_df["model_score"] = model_scores

analysis_df = analysis_df.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

# Rank produced by the model
analysis_df["model_rank"] = (
    analysis_df.index + 1
)


# ---------------------------------------------
# FALSE POSITIVES IN TOP 50
# ---------------------------------------------

top_50 = analysis_df.head(50).copy()

false_positives_top50 = top_50[
    top_50["decline_label"] == 0
].copy()

print(
    "False positives in model Top 50:",
    len(false_positives_top50)
)

display(
    false_positives_top50[
        [
            "content_id",
            "model_rank",
            "model_score",
            "decline_label",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ].head(10)
)


# ---------------------------------------------
# FALSE NEGATIVES / MISSED DECLINES
# ---------------------------------------------

missed_declines = analysis_df[
    (analysis_df["decline_label"] == 1)
    & (analysis_df["model_rank"] > 50)
].copy()

missed_declines = missed_declines.sort_values(
    "model_score",
    ascending=True
)

print(
    "\nActual decline cases outside Top 50:",
    len(missed_declines)
)

print(
    "\nLowest-scored actual decline cases:"
)

display(
    missed_declines[
        [
            "content_id",
            "model_rank",
            "model_score",
            "decline_label",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ].head(10)
)


# ---------------------------------------------
# SIMPLE SUMMARY
# ---------------------------------------------

print("\nSummary:")
print(
    f"Top-50 correct decline cases: "
    f"{int(top_50['decline_label'].sum())}/50"
)

print(
    f"Top-50 false positives: "
    f"{len(false_positives_top50)}"
)

print(
    f"Precision@50: "
    f"{top_50['decline_label'].mean():.3f}"
)

False positives in model Top 50: 14


,content_id,model_rank,model_score,decline_label,impressions_90d,ctr,avg_position,days_since_last_update
4,content_374e795aab68,5,0.900690,0,235,0.85,31.0,20
6,content_41baf0722ad9,7,0.893844,0,3115,0.00,12.8,104
7,content_7be5f150dc65,8,0.893454,0,290,0.00,5.9,20
14,content_d10f9ce1e0cd,15,0.879786,0,166,0.60,16.1,104
16,content_f0d98be4b42c,17,0.874211,0,5818,0.15,5.1,104
18,content_ce59581533ca,19,0.872736,0,289,0.69,18.8,102
23,content_f45787e64ac2,24,0.871271,0,291,0.34,5.2,104
26,content_4d9f36001f06,27,0.867772,0,3369,0.03,13.2,104
28,content_c94a53e3bfb8,29,0.866189,0,2164,0.23,8.1,20
37,content_500bd3907331,38,0.862694,0,4037,0.10,5.5,104



Actual decline cases outside Top 50: 3113

Lowest-scored actual decline cases:


,content_id,model_rank,model_score,decline_label,impressions_90d,ctr,avg_position,days_since_last_update
6105,content_7bc32bc1df59,6106,0.017640,1,1,0.00,NaN,92
6099,content_917fc1b11fe1,6100,0.084636,1,916,0.00,78.6,22
6097,content_8818fd6d967f,6098,0.087673,1,83603,1.06,3.4,104
6096,content_742a8fcba2fe,6097,0.087772,1,643,0.00,76.0,20
6085,content_8ff857ae67d0,6086,0.111680,1,21103,1.11,4.2,22
6081,content_d987db4e38fe,6082,0.119480,1,97,0.00,70.3,20
6080,content_b2beaf2fc81c,6081,0.119813,1,767,0.00,53.4,22
6073,content_7a82fa0b634e,6074,0.131545,1,41,0.00,58.2,22
6071,content_19415130d5ee,6072,0.134624,1,1021,0.00,62.1,22
6070,content_2f002563e9cd,6071,0.135563,1,17,0.00,5.2,20



Summary:
Top-50 correct decline cases: 36/50
Top-50 false positives: 14
Precision@50: 0.720


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.